<a href="https://colab.research.google.com/github/sanam-zainab/habitforge/blob/main/DSAtutorModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q unsloth transformers datasets accelerate peft bitsandbytes trl

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2",
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
)

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
from datasets import Dataset

data = [
    {"instruction": "Explain binary search", "output": "Binary search is an efficient algorithm that works on sorted arrays by repeatedly dividing the search space in half."},
    {"instruction": "What is a stack?", "output": "A stack is a linear data structure that follows Last In First Out (LIFO)."},
    {"instruction": "Explain recursion", "output": "Recursion is a method where a function calls itself to solve smaller instances of a problem."},
    {"instruction": "What is a queue?", "output": "A queue is a linear data structure that follows First In First Out (FIFO)."},
]

dataset = Dataset.from_list(data)

In [6]:
def format_data(example):
    return {
        "text": f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"
    }

dataset = dataset.map(format_data)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [9]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=2048,
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 100,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 10,
    output_dir = "outputs",
    optim = "adamw_8bit",
    save_steps = 50,              # Save checkpoint midway
    save_total_limit = 2,
    report_to = "none",
)

trainer = Trainer(
    model = model,
    train_dataset = dataset,
    args = training_args,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 100 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
10,456.450195
20,299.150635
30,295.078418
40,293.900537
50,293.347144
60,292.999438
70,292.788940
80,292.683105
90,292.627759
100,292.600562


TrainOutput(global_step=100, training_loss=310.16267333984376, metrics={'train_runtime': 2336.6591, 'train_samples_per_second': 0.342, 'train_steps_per_second': 0.043, 'total_flos': 3.51564749340672e+16, 'train_loss': 310.16267333984376, 'epoch': 100.0})

In [11]:
model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

Unsloth: Restored added_tokens_decoder metadata in fine_tuned_model/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in fine_tuned_model.


('fine_tuned_model/tokenizer_config.json',
 'fine_tuned_model/chat_template.jinja',
 'fine_tuned_model/tokenizer.json')

In [12]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r fine_tuned_model /content/drive/MyDrive/

Mounted at /content/drive
